In [2]:
import cv2
import os
from moviepy.video.io.VideoFileClip import VideoFileClip
import pandas as pd

In [20]:
import cv2
import os
from moviepy.video.io.VideoFileClip import VideoFileClip



# Load the video file and print out some basic information about it
video_path = "data/video/005_C1_C2_C3_2026-05-07.mp4"
clip = VideoFileClip(video_path)
print(clip.duration, clip.fps, clip.size)


cap = cv2.VideoCapture(video_path)

i = 0
while True:
    # ret = True if the frame is read correctly, False if there are no more frames to read
    # frame = the actual frame that was read from the video
    ret, frame = cap.read()

    if not ret:
        break
	
    cv2.imwrite("data/video_frames/005_C1_C2_C3_frames/frame_{}.jpg".format(i), frame)
    i += 1

cap.release()

4346.63 30.0 [2048, 1536]


In [3]:
# Count the number of extracted frames
video_path = "data/video/005_C1_C2_C3_2026-05-07.mp4"
clip = VideoFileClip(video_path)
frames_dir = "data/video_frames/005_C1_C2_C3_frames"
frame_files = [f for f in os.listdir(frames_dir) if f.endswith(".jpg")]
num_frames = len(frame_files)

# Calculate precise FPS from the actual frame count and video duration
video_duration_s = clip.duration  # in seconds, from the VideoFileClip loaded above
precise_fps = num_frames / video_duration_s

print(f"Frames extracted : {num_frames}")
print(f"Video duration   : {video_duration_s:.4f} s")
print(f"Reported FPS     : {clip.fps} Hz")
print(f"Precise FPS      : {precise_fps:.6f} Hz")

Frames extracted : 130399
Video duration   : 4346.6300 s
Reported FPS     : 30.0 Hz
Precise FPS      : 30.000023 Hz


In [4]:
df = pd.read_parquet("data/merged/005_synced_with_objects.parquet")

In [5]:
df

,gaze_capture_time,raw_timestamp,relative_to_unix_epoch_timestamp,focus_distance,frame_number,stability,status,gaze_forward_x,gaze_forward_y,gaze_forward_z,...,bt_is_interpolated_LeftHand,bt_bad_sample_RightHand,bt_is_interpolated_RightHand,bt_segment_label,saccade_id,fixation_id,event_duration_ms,fixation_target,lefthand_grab_id,righthand_grab_id
0,1000005717474053800,1778146382884,919.4069,0.408554,573336,0.000000,Valid,-0.005948,-0.222977,0.974805,...,NaN,NaN,NaN,None,<NA>,567,415.058432,NextItemButton,<NA>,<NA>
1,1000005717479054300,1778146382884,919.4069,0.410998,573337,0.000000,Valid,-0.005909,-0.224134,0.974540,...,NaN,NaN,NaN,None,<NA>,567,415.058432,NextItemButton,<NA>,<NA>
2,1000005717574065800,1778146382885,919.4069,0.442411,573356,0.448309,Valid,-0.011875,-0.235372,0.971833,...,NaN,NaN,NaN,None,<NA>,567,415.058432,NextItemButton,<NA>,<NA>
3,1000005717579067800,1778146382885,919.4069,0.434882,573357,0.435031,Valid,-0.012998,-0.234836,0.971948,...,NaN,NaN,NaN,None,<NA>,567,415.058432,NextItemButton,<NA>,<NA>
4,1000005717584068400,1778146382885,919.4069,0.425938,573358,0.468381,Valid,-0.013930,-0.234515,0.972013,...,NaN,NaN,NaN,None,<NA>,567,415.058432,NextItemButton,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
567593,1000009306851704400,1778149972036,3949.8870,0.713750,1290935,0.000000,Valid,0.143717,-0.185808,0.972019,...,NaN,NaN,NaN,None,<NA>,18883,65.009152,work_desk_base,<NA>,<NA>
567594,1000009306856705400,1778149972047,3949.8980,0.715080,1290936,0.000000,Valid,0.143869,-0.187288,0.971712,...,NaN,NaN,NaN,None,<NA>,18883,65.009152,work_desk_base,<NA>,<NA>
567595,1000009306861706400,1778149972047,3949.8980,0.711939,1290937,0.000000,Valid,0.143650,-0.188271,0.971555,...,NaN,NaN,NaN,None,<NA>,18883,65.009152,work_desk_base,<NA>,<NA>
567596,1000009306866707500,1778149972058,3949.9090,0.706908,1290938,0.000000,Valid,0.143264,-0.189703,0.971333,...,NaN,NaN,NaN,None,<NA>,18883,65.009152,work_desk_base,<NA>,<NA>


In [6]:
def build_model_duration_dict(df):
    # A new build starts when model_name switches from NaN to an actual name
    prev_missing = df['model_name'].isna()
    df['_building_start'] = df['model_name'].notna() & prev_missing.shift(1, fill_value=True)

    # Create a group ID that increments only at each real transition
    df['_group_id'] = df['_building_start'].cumsum()

    # Keep only rows where a model name is present
    building_df = df[df['model_name'].notna()].copy()

    # Group by the group ID and aggregate
    grouped = (
        building_df
        .groupby('_group_id')
        .agg(
            model_name=('model_name', 'first'),
            start_idx=('_group_id', lambda x: x.index[0]),
            end_idx=('_group_id', lambda x: x.index[-1]),
            row_count=('_group_id', 'count')
        )
        .reset_index(drop=True)
    )

    # Handle TM renaming (TM1, TM2, TM3, ...)
    tm_counter = 0
    model_dict = {}

    for _, row in grouped.iterrows():
        name = row['model_name']
        if name == 'TM':
            tm_counter += 1
            key = f'TM{tm_counter}'
        else:
            key = name

        model_dict[key] = {
            'row_count': row['row_count'],
            'start_idx': row['start_idx'],
            'end_idx': row['end_idx']
        }

    # Clean up temporary columns
    df.drop(columns=['_building_start', '_group_id'], inplace=True)

    return model_dict


model_durations = build_model_duration_dict(df)

# Output
for model, info in model_durations.items():
    print(f"{model}: {info['row_count']} rows (index {info['start_idx']} – {info['end_idx']})")

TM1: 19676 rows (index 2990 – 22665)
C1M3F: 9250 rows (index 32386 – 41635)
C3M3F: 26396 rows (index 48379 – 74774)
C2M3F: 16811 rows (index 88978 – 105788)
C2M6A: 14086 rows (index 112470 – 126555)
C3M2A: 33147 rows (index 132722 – 165868)
C1M6A: 8715 rows (index 172470 – 181184)
TM2: 20892 rows (index 190158 – 211049)
C2M4A: 25959 rows (index 217295 – 243253)
C3M5F: 29775 rows (index 248983 – 278757)
C1M4A: 9006 rows (index 284837 – 293842)
C1M1F: 7945 rows (index 300979 – 308923)
C2M5F: 16863 rows (index 314509 – 331371)
C3M6A: 31991 rows (index 337173 – 369163)
TM3: 21279 rows (index 386427 – 407705)
C3M1F: 33975 rows (index 412902 – 446876)
C1M2A: 9389 rows (index 452176 – 461564)
C2M2A: 20933 rows (index 467408 – 488340)
C3M4A: 30830 rows (index 493728 – 524557)
C1M5F: 7215 rows (index 530108 – 537322)
C2M1F: 16603 rows (index 542483 – 559085)


In [7]:
import subprocess, json

# Videostart-Zeit aus Metadaten lesen (funktioniert wenn die Kamera einen Timestamp einbettet)
result = subprocess.run(
    ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_format", "data/video/005_C1_C2_C3_2026-05-07.mp4"],
    capture_output=True, text=True
)
meta = json.loads(result.stdout)
video_creation_time = meta["format"]["tags"].get("creation_time")  # z.B. "2024-03-15T10:23:45Z"
print(video_creation_time)

2026-05-07T09:20:36.000000Z


In [3]:
import argparse
import json
import math
import shutil
import subprocess
import sys
from datetime import datetime, timezone
 
import numpy as np
import pandas as pd


In [9]:
def _parse_creation_time(value: str) -> float:
    """Return Unix epoch milliseconds for an ISO-8601 creation_time string."""
    s = value.strip().replace("Z", "+00:00")
    dt = datetime.fromisoformat(s)
    if dt.tzinfo is None:                      # assume UTC if no offset given
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.timestamp() * 1000.0

In [10]:
def read_video_metadata(video_path):
    """
    Return (creation_ms, fps, duration_s). Any field may be None if it could
    not be read; the caller can fill gaps from CLI overrides.
 
    Robust against the "format errors" seen with some files: we try ffprobe
    first (works on .mp4/.avi/.mkv) and fall back to moviepy for fps/duration.
    """
    creation_ms = fps = duration_s = None
 
    if shutil.which("ffprobe"):
        try:
            out = subprocess.run(
                ["ffprobe", "-v", "quiet", "-print_format", "json",
                 "-show_format", "-show_streams", video_path],
                capture_output=True, text=True, check=False,
            ).stdout
            meta = json.loads(out) if out.strip() else {}
 
            fmt = meta.get("format", {})
            tags = fmt.get("tags", {}) or {}
            ct = tags.get("creation_time")
            if ct:
                creation_ms = _parse_creation_time(ct)
            if fmt.get("duration"):
                duration_s = float(fmt["duration"])
 
            for st in meta.get("streams", []):
                if st.get("codec_type") == "video":
                    # creation_time is sometimes only on the stream
                    if creation_ms is None:
                        sct = (st.get("tags", {}) or {}).get("creation_time")
                        if sct:
                            creation_ms = _parse_creation_time(sct)
                    rate = st.get("avg_frame_rate") or st.get("r_frame_rate")
                    if rate and rate not in ("0/0", "0"):
                        num, _, den = rate.partition("/")
                        den = den or "1"
                        if float(den) != 0:
                            fps = float(num) / float(den)
                    if duration_s is None and st.get("duration"):
                        duration_s = float(st["duration"])
                    break
        except Exception as exc:                       # never hard-fail here
            print(f"[warn] ffprobe parsing failed: {exc}", file=sys.stderr)
 
    if (fps is None or duration_s is None):
        try:
            from moviepy.video.io.VideoFileClip import VideoFileClip
            clip = VideoFileClip(video_path)
            fps = fps or float(clip.fps)
            duration_s = duration_s or float(clip.duration)
            clip.close()
        except Exception as exc:
            print(f"[warn] moviepy fallback failed: {exc}", file=sys.stderr)
 
    return creation_ms, fps, duration_s

In [11]:
creation_ms, fps, duration_s =read_video_metadata("data/video/005_C1_C2_C3_2026-05-07.mp4")

In [12]:
def add_columns(df, creation_ms, fps, duration_s,
                ns_col="gaze_capture_time",
                unix_ms_col="raw_timestamp",
                offset_seconds=0.0):
    if ns_col not in df.columns:
        raise KeyError(f"Column '{ns_col}' (Varjo monotonic ns) not found.")
    if unix_ms_col not in df.columns:
        raise KeyError(f"Column '{unix_ms_col}' (Unix ms) not found.")
 
    ns = pd.to_numeric(df[ns_col], errors="coerce").astype("float64")
    unix_ms = pd.to_numeric(df[unix_ms_col], errors="coerce").astype("float64")
 
    # Use the earliest valid row to bridge Unix-ms -> Varjo-ns near frame 0.
    valid = ns.notna() & unix_ms.notna()
    if not valid.any():
        raise ValueError("No rows with both timestamps present.")
    i0 = valid.idxmax()
    first_ns = ns.loc[i0]
    first_unix_ms = unix_ms.loc[i0]
 
    # Express video frame-0 in the monotonic ns domain, plus optional manual nudge.
    frame0_ns = first_ns - (first_unix_ms - creation_ms) * 1e6 + offset_seconds * 1e9
 
    t_rel_s = (ns - frame0_ns) / 1e9
    frame_idx = np.floor(t_rel_s * fps)
 
    df = df.copy()
    df["video_relative_timestamp_s"] = t_rel_s
    df["video_frame_index"] = frame_idx.astype("Int64")
 
    # Diagnostics ----------------------------------------------------------- #
    n_total = len(df)
    n_before = int((t_rel_s < 0).sum())
    last_frame = (math.floor(duration_s * fps) - 1) if duration_s else None
    n_after = int((frame_idx > last_frame).sum()) if last_frame is not None else 0
 
    print("=" * 64)
    print("ALIGNMENT SUMMARY")
    print("=" * 64)
    print(f"rows                         : {n_total:,}")
    print(f"video fps                    : {fps}")
    if duration_s:
        print(f"video duration               : {duration_s:.3f} s "
              f"({duration_s/60:.2f} min), last frame index {last_frame}")
    vid0 = datetime.fromtimestamp(creation_ms/1000, tz=timezone.utc)
    print(f"video frame-0 wall clock     : {vid0:%Y-%m-%d %H:%M:%S.%f} UTC")
    print(f"ET starts in video at        : {t_rel_s[valid].min():.3f} s "
          f"({t_rel_s[valid].min()/60:.2f} min)  -> frame "
          f"{int(np.floor(t_rel_s[valid].min()*fps))}")
    print(f"ET ends in video at          : {t_rel_s[valid].max():.3f} s "
          f"({t_rel_s[valid].max()/60:.2f} min)  -> frame "
          f"{int(np.floor(t_rel_s[valid].max()*fps))}")
    print(f"samples before video start   : {n_before}")
    if duration_s:
        print(f"samples after  video end     : {n_after}")
    if n_before or n_after:
        print("  [check] non-zero out-of-range counts can mean the wrong "
              "creation_time, a clipped video, or a needed --offset-seconds.")
    # report observed clock drift for transparency
    last_valid = valid[::-1].idxmax()
    off0 = (unix_ms.loc[i0] * 1e6 - ns.loc[i0]) / 1e9
    off1 = (unix_ms.loc[last_valid] * 1e6 - ns.loc[last_valid]) / 1e9
    print(f"OS-vs-monotonic clock drift  : {(off1-off0)*1000:.1f} ms across recording")
    print("=" * 64)
 
    return df

In [13]:
df = add_columns(df, creation_ms, fps, duration_s)

ALIGNMENT SUMMARY
rows                         : 567,598
video fps                    : 30.000029908234634
video duration               : 4346.629 s (72.44 min), last frame index 130397
video frame-0 wall clock     : 2026-05-07 09:20:36.000000 UTC
ET starts in video at        : 746.884 s (12.45 min)  -> frame 22406
ET ends in video at          : 4336.282 s (72.27 min)  -> frame 130088
samples before video start   : 0
samples after  video end     : 0
OS-vs-monotonic clock drift  : -223.7 ms across recording


In [14]:
df.to_parquet("data/merged/005_synced_with_video.parquet", index=False)

In [15]:
# Keep only rows where model_name is present AND is not the tutorial build "TM"
df_models = df[df["model_name"].notna() & (df["model_name"] != "TM")].copy()

In [16]:
df_models

,gaze_capture_time,raw_timestamp,relative_to_unix_epoch_timestamp,focus_distance,frame_number,stability,status,gaze_forward_x,gaze_forward_y,gaze_forward_z,...,bt_is_interpolated_RightHand,bt_segment_label,saccade_id,fixation_id,event_duration_ms,fixation_target,lefthand_grab_id,righthand_grab_id,video_relative_timestamp_s,video_frame_index
32386,1000005957131472600,1778146622321,1158.836,0.913428,621249,0.775880,Valid,-0.037878,0.016019,0.999154,...,0.0,C1_T1,<NA>,689,295.041536,Wall_Front,<NA>,1,986.541419,29596
32387,1000005957136473000,1778146622321,1158.836,0.912220,621250,0.779513,Valid,-0.037908,0.016023,0.999153,...,0.0,C1_T1,<NA>,689,295.041536,Wall_Front,<NA>,1,986.546419,29596
32388,1000005957126472200,1778146622321,1158.836,0.914912,621248,0.777160,Valid,-0.037847,0.016018,0.999155,...,0.0,C1_T1,<NA>,689,295.041536,Wall_Front,<NA>,1,986.536418,29596
32389,1000005957141474300,1778146622332,1158.846,0.911166,621251,0.780521,Valid,-0.037940,0.016031,0.999151,...,0.0,C1_T1,<NA>,689,295.041536,Wall_Front,<NA>,1,986.551421,29596
32390,1000005957146475300,1778146622332,1158.846,0.910019,621252,0.797688,Valid,-0.037966,0.016049,0.999150,...,0.0,C1_T1,<NA>,689,295.041536,Wall_Front,<NA>,1,986.556422,29596
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
559081,1000009262545079000,1778149927817,3905.665,2.000000,1282077,0.494065,Valid,0.424640,-0.177512,0.887790,...,0.0,C3_T6,<NA>,18057,1355.190784,NextItemButton,<NA>,321,4291.955025,128758
559082,1000009262525077900,1778149927817,3905.665,2.000000,1282073,0.487071,Valid,0.424842,-0.180462,0.887098,...,0.0,C3_T6,<NA>,18057,1355.190784,NextItemButton,<NA>,321,4291.935024,128758
559083,1000009262620087400,1778149927818,3905.665,2.000000,1282092,0.538331,Valid,0.422669,-0.176345,0.888962,...,0.0,C3_T6,<NA>,18057,1355.190784,NextItemButton,<NA>,321,4292.030034,128761
559084,1000009262625086800,1778149927818,3905.665,2.000000,1282093,0.531270,Valid,0.422527,-0.176555,0.888988,...,0.0,C3_T6,<NA>,18057,1355.190784,NextItemButton,<NA>,321,4292.035033,128761


In [19]:
df_models = df_models.sort_values("gaze_capture_time",)
df_models

,gaze_capture_time,raw_timestamp,relative_to_unix_epoch_timestamp,focus_distance,frame_number,stability,status,gaze_forward_x,gaze_forward_y,gaze_forward_z,...,bt_is_interpolated_RightHand,bt_segment_label,saccade_id,fixation_id,event_duration_ms,fixation_target,lefthand_grab_id,righthand_grab_id,video_relative_timestamp_s,video_frame_index
32388,1000005957126472200,1778146622321,1158.836,0.914912,621248,0.777160,Valid,-0.037847,0.016018,0.999155,...,0.0,C1_T1,<NA>,689,295.041536,Wall_Front,<NA>,1,986.536418,29596
32386,1000005957131472600,1778146622321,1158.836,0.913428,621249,0.775880,Valid,-0.037878,0.016019,0.999154,...,0.0,C1_T1,<NA>,689,295.041536,Wall_Front,<NA>,1,986.541419,29596
32387,1000005957136473000,1778146622321,1158.836,0.912220,621250,0.779513,Valid,-0.037908,0.016023,0.999153,...,0.0,C1_T1,<NA>,689,295.041536,Wall_Front,<NA>,1,986.546419,29596
32389,1000005957141474300,1778146622332,1158.846,0.911166,621251,0.780521,Valid,-0.037940,0.016031,0.999151,...,0.0,C1_T1,<NA>,689,295.041536,Wall_Front,<NA>,1,986.551421,29596
32390,1000005957146475300,1778146622332,1158.846,0.910019,621252,0.797688,Valid,-0.037966,0.016049,0.999150,...,0.0,C1_T1,<NA>,689,295.041536,Wall_Front,<NA>,1,986.556422,29596
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
559064,1000009262610082900,1778149927817,3905.665,2.000000,1282090,0.481867,Valid,0.423000,-0.176025,0.888868,...,0.0,C3_T6,<NA>,18057,1355.190784,NextItemButton,<NA>,321,4292.020029,128760
559067,1000009262615085200,1778149927817,3905.665,2.000000,1282091,0.490450,Valid,0.422825,-0.176173,0.888922,...,0.0,C3_T6,<NA>,18057,1355.190784,NextItemButton,<NA>,321,4292.025031,128760
559083,1000009262620087400,1778149927818,3905.665,2.000000,1282092,0.538331,Valid,0.422669,-0.176345,0.888962,...,0.0,C3_T6,<NA>,18057,1355.190784,NextItemButton,<NA>,321,4292.030034,128761
559084,1000009262625086800,1778149927818,3905.665,2.000000,1282093,0.531270,Valid,0.422527,-0.176555,0.888988,...,0.0,C3_T6,<NA>,18057,1355.190784,NextItemButton,<NA>,321,4292.035033,128761


In [21]:
import cv2, os
from collections import defaultdict

# ---------------------------------------------------------------------------
# Fast, direct extraction of building-phase frames straight from the video.
# No intermediate "dump all 130k frames" step: decode the video ONCE and only
# WRITE frames that fall inside a building phase (+/- 2 s buffer).
# ---------------------------------------------------------------------------

VIDEO_PATH   = "data/video/005_C1_C2_C3_2026-05-07.mp4"
FRAMES_ROOT  = "data/video_frames"
SESSION      = "005"
PAD_FRAMES   = 60     # 2 s * 30 fps, added before AND after each phase
JPEG_QUALITY = 90     # 1-100; lower = smaller files & faster writes (95 = cv2 default)

# --- 1. One row per building phase (each model is built exactly once) --------
phases = (df_models.groupby("model_name", as_index=False)
          .agg(condition=("condition_number", "first"),
               start_frame=("video_frame_index", "min"),
               end_frame=("video_frame_index", "max")))

# Number phases B1..B6 within each condition, in chronological order
phases = phases.sort_values(["condition", "start_frame"]).reset_index(drop=True)
phases["B"] = phases.groupby("condition").cumcount() + 1
print(phases[["condition", "B", "model_name", "start_frame", "end_frame"]].to_string(index=False))
assert len(phases) == 18, f"expected 18 phases, got {len(phases)} -- check df_models"

# --- 2. Map each needed frame index -> the folder(s) that want it ------------
# (a frame can fall in the buffers of two neighbouring phases)
frame_to_folders = defaultdict(list)
for _, p in phases.iterrows():
    folder = os.path.join(FRAMES_ROOT, f"{SESSION}_C{int(p.condition)}_B{int(p.B)}")
    os.makedirs(folder, exist_ok=True)
    lo = max(0, int(p.start_frame) - PAD_FRAMES)   # don't go below frame 0
    hi = int(p.end_frame) + PAD_FRAMES             # frames past the end are simply never reached
    for i in range(lo, hi + 1):
        frame_to_folders[i].append(folder)

# --- 3. Decode the video ONCE; retrieve + write only the frames we need ------
cap = cv2.VideoCapture(VIDEO_PATH)
enc = [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY]
idx, written = 0, 0
while True:
    if not cap.grab():            # advance decoder (cheap); False = end of video
        break
    if idx in frame_to_folders:   # this frame belongs to >= 1 building phase
        ok, frame = cap.retrieve()   # expensive step, done only when needed
        if ok:
            fname = f"frame_{idx}.jpg"          # keep the ORIGINAL global frame index
            for folder in frame_to_folders[idx]:
                cv2.imwrite(os.path.join(folder, fname), frame, enc)
                written += 1
    idx += 1
    if idx % 20000 == 0:
        print(f"  ...scanned {idx} frames, written {written}")
cap.release()

print(f"\nDone: scanned {idx} frames, wrote {written} files across {len(phases)} folders.")

 condition  B model_name  start_frame  end_frame
         1  1      C1M3F        29596      31072
         1  2      C3M3F        32121      36336
         1  3      C2M3F        38576      41263
         1  4      C2M6A        42328      44574
         1  5      C3M2A        45525      50690
         1  6      C1M6A        51723      53084
         2  1      C2M4A        69570      73594
         2  2      C3M5F        74484      79063
         2  3      C1M4A        79994      81393
         2  4      C1M1F        82487      83713
         2  5      C2M5F        84575      87169
         2  6      C3M6A        88065      93000
         3  1      C3M1F       106158     111419
         3  2      C1M2A       112251     113724
         3  3      C2M2A       114637     117847
         3  4      C3M4A       118688     123411
         3  5      C1M5F       124277     125392
         3  6      C2M1F       126196     128761
  ...scanned 20000 frames, written 0
  ...scanned 40000 frames, writt

In [4]:
df_sorted = pd.read_parquet("data/merged/005_synced_with_video.parquet")
df_sorted = df_sorted.sort_values("gaze_capture_time")
df_sorted

,gaze_capture_time,raw_timestamp,relative_to_unix_epoch_timestamp,focus_distance,frame_number,stability,status,gaze_forward_x,gaze_forward_y,gaze_forward_z,...,bt_is_interpolated_RightHand,bt_segment_label,saccade_id,fixation_id,event_duration_ms,fixation_target,lefthand_grab_id,righthand_grab_id,video_relative_timestamp_s,video_frame_index
0,1000005717474053800,1778146382884,919.4069,0.408554,573336,0.0,Valid,-0.005948,-0.222977,0.974805,...,NaN,None,<NA>,567,415.058432,NextItemButton,<NA>,<NA>,746.884000,22406
1,1000005717479054300,1778146382884,919.4069,0.410998,573337,0.0,Valid,-0.005909,-0.224134,0.974540,...,NaN,None,<NA>,567,415.058432,NextItemButton,<NA>,<NA>,746.889001,22406
19,1000005717484054300,1778146382885,919.4069,0.408002,573338,0.0,Valid,-0.005868,-0.224828,0.974381,...,NaN,None,<NA>,569,90.012672,work_desk_base,<NA>,<NA>,746.894001,22406
20,1000005717489054500,1778146382885,919.4069,0.411857,573339,0.0,Valid,-0.005935,-0.225239,0.974285,...,NaN,None,<NA>,569,90.012672,work_desk_base,<NA>,<NA>,746.899001,22406
21,1000005717494061000,1778146382885,919.4069,0.411196,573340,0.0,Valid,-0.005951,-0.225754,0.974166,...,NaN,None,<NA>,569,90.012672,work_desk_base,<NA>,<NA>,746.904007,22407
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
567593,1000009306851704400,1778149972036,3949.8870,0.713750,1290935,0.0,Valid,0.143717,-0.185808,0.972019,...,NaN,None,<NA>,18883,65.009152,work_desk_base,<NA>,<NA>,4336.261651,130087
567594,1000009306856705400,1778149972047,3949.8980,0.715080,1290936,0.0,Valid,0.143869,-0.187288,0.971712,...,NaN,None,<NA>,18883,65.009152,work_desk_base,<NA>,<NA>,4336.266652,130088
567595,1000009306861706400,1778149972047,3949.8980,0.711939,1290937,0.0,Valid,0.143650,-0.188271,0.971555,...,NaN,None,<NA>,18883,65.009152,work_desk_base,<NA>,<NA>,4336.271653,130088
567596,1000009306866707500,1778149972058,3949.9090,0.706908,1290938,0.0,Valid,0.143264,-0.189703,0.971333,...,NaN,None,<NA>,18883,65.009152,work_desk_base,<NA>,<NA>,4336.276654,130088


In [9]:
DF1 = df_sorted[["video_frame_index", "hit_obj_name", "event_type"]]
DF1

,video_frame_index,hit_obj_name,event_type
0,22406,work_desk_base,fixation
1,22406,work_desk_base,fixation
19,22406,work_desk_base,fixation
20,22406,work_desk_base,fixation
21,22407,work_desk_base,fixation
...,...,...,...
567593,130087,work_desk_base,fixation
567594,130088,work_desk_base,fixation
567595,130088,work_desk_base,fixation
567596,130088,work_desk_base,fixation


In [5]:
# This cell creates an additional column in the dataset containing the correct frame for each
# datapoint in building phase 1
# (used later for precise mapping of eye-tracking data to video-data)


import numpy as np, pandas as pd

DF = df_sorted
PHASE_MODEL = "C1M3F"        # Building Phase 1 (Condition 1, B1)

# --- zwei am Video abgelesene Anker -----------------------------------------
START_TRUE = 29578           # Frame bei time_ms == 0   (aktuell 29596)
AFTER_TRUE = 31059           # erster Frame NACH der Phase (wo time_ms -> NaN)

# --- Phase 1 und die Zeile direkt danach lokalisieren -----------------------
mask = DF["model_name"] == PHASE_MODEL
pos  = np.where(mask.to_numpy())[0]
start_pos, end_pos = pos[0], pos[-1]

start_old = int(DF["video_frame_index"].iloc[start_pos])      # aktueller Frame bei time_ms==0
after_old = int(DF["video_frame_index"].iloc[end_pos + 1])    # aktueller Frame der 1. NaN-Zeile danach

# --- lineares Remapping: aktueller Frame -> korrigierter Frame ---------------
# Die zwei Anker (Start + erster Frame danach) legen Versatz UND Steigung fest.
slope = (AFTER_TRUE - START_TRUE) / (after_old - start_old)

DF["correct_frame"] = pd.Series(pd.NA, index=DF.index, dtype="Int64")  # alles andere = NaN
DF.loc[mask, "correct_frame"] = np.round(
    START_TRUE + (DF.loc[mask, "video_frame_index"] - start_old) * slope
).astype("Int64")

# --- Diagnose ---------------------------------------------------------------
last_time_ms = float(DF["time_ms"].iloc[end_pos])
print(f"Anker:  ({start_old} -> {START_TRUE})   ({after_old} -> {AFTER_TRUE})")
print(f"Versatz am Start : {START_TRUE - start_old} Frames   |   slope = {slope:.6f}")
print(f"correct_frame Start/Ende der Phase : "
      f"{DF.loc[mask,'correct_frame'].min()} / {DF.loc[mask,'correct_frame'].max()}")
print(f"letztes time_ms der Phase : {last_time_ms:.0f} ms ({last_time_ms/1000:.2f} s)")
print(f"Frame-Spanne der Phase    : {after_old - start_old} Frames "
      f"({(after_old - start_old)/30:.2f} s bei 30 fps)")

Anker:  (29596 -> 29578)   (31072 -> 31059)
Versatz am Start : -18 Frames   |   slope = 1.003388
correct_frame Start/Ende der Phase : 29578 / 31059
letztes time_ms der Phase : 49199 ms (49.20 s)
Frame-Spanne der Phase    : 1476 Frames (49.20 s bei 30 fps)


In [6]:
building_phase_1 = df_sorted[df_sorted["model_name"] == "C1M3F"].copy()
building_phase_1

,gaze_capture_time,raw_timestamp,relative_to_unix_epoch_timestamp,focus_distance,frame_number,stability,status,gaze_forward_x,gaze_forward_y,gaze_forward_z,...,bt_segment_label,saccade_id,fixation_id,event_duration_ms,fixation_target,lefthand_grab_id,righthand_grab_id,video_relative_timestamp_s,video_frame_index,correct_frame
32388,1000005957126472200,1778146622321,1158.836,0.914912,621248,0.777160,Valid,-0.037847,0.016018,0.999155,...,C1_T1,<NA>,689,295.041536,Wall_Front,<NA>,1,986.536418,29596,29578
32386,1000005957131472600,1778146622321,1158.836,0.913428,621249,0.775880,Valid,-0.037878,0.016019,0.999154,...,C1_T1,<NA>,689,295.041536,Wall_Front,<NA>,1,986.541419,29596,29578
32387,1000005957136473000,1778146622321,1158.836,0.912220,621250,0.779513,Valid,-0.037908,0.016023,0.999153,...,C1_T1,<NA>,689,295.041536,Wall_Front,<NA>,1,986.546419,29596,29578
32389,1000005957141474300,1778146622332,1158.846,0.911166,621251,0.780521,Valid,-0.037940,0.016031,0.999151,...,C1_T1,<NA>,689,295.041536,Wall_Front,<NA>,1,986.551421,29596,29578
32390,1000005957146475300,1778146622332,1158.846,0.910019,621252,0.797688,Valid,-0.037966,0.016049,0.999150,...,C1_T1,<NA>,689,295.041536,Wall_Front,<NA>,1,986.556422,29596,29578
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41615,1000006006305713600,1778146671513,1208.026,0.568300,631080,0.013834,Valid,0.300425,-0.123471,0.945780,...,C1_T1,<NA>,906,2480.349184,work_desk_base,<NA>,7,1035.715660,31071,31058
41616,1000006006310715100,1778146671513,1208.026,0.571531,631081,0.000000,Valid,0.300356,-0.123701,0.945772,...,C1_T1,<NA>,906,2480.349184,work_desk_base,<NA>,7,1035.720661,31071,31058
41617,1000006006315716700,1778146671513,1208.026,0.571512,631082,0.000000,Valid,0.300696,-0.123807,0.945650,...,C1_T1,<NA>,906,2480.349184,work_desk_base,<NA>,7,1035.725663,31071,31058
41620,1000006006320718200,1778146671513,1208.026,0.574508,631083,0.000000,Valid,0.300620,-0.124061,0.945641,...,C1_T1,<NA>,906,2480.349184,work_desk_base,<NA>,7,1035.730664,31071,31058


In [7]:
import os, re, cv2, pandas as pd

SRC_DIR = "data/video_frames/005_C1_B1"
DST_DIR = "data/video_frames/005_C1_B1_wobjects"
os.makedirs(DST_DIR, exist_ok=True)

def valid(o):
    return pd.notna(o) and str(o).strip() != ""

def obj_mode(s):                                  # most frequent valid object in the frame
    s = s[s.map(valid)]
    return s.mode().iloc[0] if not s.mode().empty else pd.NA

def evt_mode(s):                                  # most frequent event_type in the frame
    return s.mode().iloc[0] if not s.mode().empty else "fixation"

# 1. Relevant rows (building phase 1)
sub = df_sorted[df_sorted["model_name"] == "C1M3F"].dropna(subset=["correct_frame"]).copy()

# 2. Per frame: most frequent object + most frequent event_type
per_frame = (sub.groupby("correct_frame")
                .agg(obj=("hit_obj_name", obj_mode), evt=("event_type", evt_mode))
                .sort_index())

# 3. Running display state over frames: update only on saccade + object change
display, current = {}, None
for frame_idx, row in per_frame.iterrows():
    obj, evt = row["obj"], row["evt"]
    if current is None and valid(obj):
        current = obj                             # initial value
    if evt == "saccade" and valid(obj) and obj != current:
        current = obj                             # update only on saccade when the object changes
    display[int(frame_idx)] = current if valid(current) else ""

# 4. Copy ALL frames in the folder + draw the box (buffer frames = empty)
FONT, SCALE, THICK, PAD, MARGIN, MIN_W = cv2.FONT_HERSHEY_SIMPLEX, 2.2, 4, 22, 40, 150
(_, ref_th), ref_base = cv2.getTextSize("Ag", FONT, SCALE, THICK)   # fixed box height
box_h = ref_th + ref_base + 2*PAD

n = 0
for fname in sorted(os.listdir(SRC_DIR)):
    m = re.fullmatch(r"frame_(\d+)\.jpg", fname)
    if not m:
        continue
    idx   = int(m.group(1))
    label = display.get(idx, "")                  # building phase -> object ; otherwise empty

    img = cv2.imread(os.path.join(SRC_DIR, fname))
    H, W = img.shape[:2]

    if label:
        (tw, _), _ = cv2.getTextSize(label, FONT, SCALE, THICK)
        box_w = max(MIN_W, tw + 2*PAD)
    else:
        box_w = MIN_W                             # empty box, same height

    x2, y1 = W - MARGIN, MARGIN                    # top-right corner
    x1, y2 = x2 - box_w, y1 + box_h

    overlay = img.copy()
    cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.55, img, 0.45, 0, img)
    cv2.rectangle(img, (x1, y1), (x2, y2), (255, 255, 255), 2)
    if label:
        cv2.putText(img, label, (x1+PAD, y1+PAD+ref_th), FONT, SCALE, (255,255,255), THICK, cv2.LINE_AA)

    cv2.imwrite(os.path.join(DST_DIR, fname), img)
    n += 1

print(f"{n} Frames -> {DST_DIR}")

1597 Frames -> data/video_frames/005_C1_B1_wobjects


In [8]:
import os, re, cv2

SRC_DIR  = "data/video_frames/005_C1_B1_wobjects"
OUT_PATH = "data/video/005_C1_B1_wobjects.mp4"
FPS      = 30
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

# 1. Collect frames and sort them NUMERICALLY (not lexicographically:
#    frame_2 must come before frame_10)
frames = []
for fname in os.listdir(SRC_DIR):
    m = re.fullmatch(r"frame_(\d+)\.jpg", fname)
    if m:
        frames.append((int(m.group(1)), fname))
frames.sort(key=lambda x: x[0])            # sort by the integer frame index

# 2. Take the frame size from the first image (all frames share it)
first = cv2.imread(os.path.join(SRC_DIR, frames[0][1]))
H, W = first.shape[:2]

# 3. Open the video writer and append every frame in order
writer = cv2.VideoWriter(OUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), FPS, (W, H))
for _, fname in frames:
    writer.write(cv2.imread(os.path.join(SRC_DIR, fname)))
writer.release()

print(f"Wrote {len(frames)} frames -> {OUT_PATH}  ({W}x{H} @ {FPS} fps)")

Wrote 1597 frames -> data/video/005_C1_B1_wobjects.mp4  (2048x1536 @ 30 fps)
